# VAJRA Nowcasting - Colab Training Notebook

**SIH 2026 PS26072** | Thunderstorm & Lightning Nowcasting

Run this notebook on **Google Colab (free T4 GPU)** to train the ConvLSTM model.

---

In [ ]:
# ============================================================
# 1. ENVIRONMENT CHECK
# ============================================================
import torch, sys, os, json, subprocess, shutil
from pathlib import Path

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU - enable Runtime > Change runtime type > GPU > T4")

In [ ]:
# ============================================================
# 2. CLONE REPO & INSTALL DEPS
# ============================================================
# EDIT THIS: replace with your GitHub repo URL
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
REPO_DIR = "VAJRA"

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# Install requirements
!pip install -r requirements.txt -q

# Verify key imports
import numpy as np
import sqlalchemy
print("Imports OK")

In [ ]:
# ============================================================
# 3. GET STARTER DATASET
# ============================================================
# Choose ONE method:

# --- METHOD A: Upload via Colab UI (Files panel - Upload folder) ---
# Upload 'data/starter_dataset/' folder from your local machine
# This is ~200 MB, takes 1-2 min

# --- METHOD B: Google Drive (if you have it there) ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/VAJRA/data/starter_dataset data/

# --- METHOD C: Re-generate synthetic data (slow, ~5 min) ---
# !python generate_starter_dataset.py

# Verify dataset exists
data_dir = Path("data/starter_dataset").resolve()
db_path = data_dir / "vajra.sqlite3"
print(f"Dataset dir: {data_dir}")
print(f"DB exists: {db_path.exists()}")
print(f"Grids: {len(list((data_dir / 'grids' / 'radar').glob('*.npy')))} radar frames")

if not db_path.exists():
    raise FileNotFoundError(
        "Dataset not found. Upload 'data/starter_dataset/' via Files panel or run generate_starter_dataset.py"
    )

In [ ]:
# ============================================================
# 4. VERIFY DATA PIPELINE
# ============================================================
import db
from db import DBSequenceDataset
from torch.utils.data import DataLoader

# Initialize DB
db.init_db(f"sqlite:///{db_path.as_posix()}", data_dir)

# Check splits
for split in ["train", "val", "test"]:
    ds = DBSequenceDataset(split=split)
    print(f"{split:5s}: {len(ds)} sequences")

# Test one batch
loader = DataLoader(DBSequenceDataset(split="train"), batch_size=2, shuffle=True)
X, y_r, y_l = next(iter(loader))
print(f"\nBatch shapes:")
print(f"  X: {X.shape}  (B, T_in, C, H, W)")
print(f"  y_r: {y_r.shape}  (B, T_out, H, W)  -- reflectivity targets (dBZ)")
print(f"  y_l: {y_l.shape}  (B, T_out, H, W)  -- lightning targets (0/1)")

In [ ]:
# ============================================================
# 5. MODEL & LOSS SETUP
# ============================================================
from model import NowcastNet, nowcast_loss
from normalization import load_normalization_stats, CHANNEL_ORDER

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Radar normalization stats (required by loss)
_REFLECT_CHANNEL = CHANNEL_ORDER.index("radar_dbz")
_means, _stds = load_normalization_stats()
REFLECT_MEAN = float(_means[_REFLECT_CHANNEL])
REFLECT_STD = float(_stds[_REFLECT_CHANNEL])
print(f"Radar norm: mean={REFLECT_MEAN:.3f}, std={REFLECT_STD:.3f}")

# Model
model = NowcastNet(in_channels=5, hidden_ch=32, output_steps=18).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {n_params:,}")

In [ ]:
# ============================================================
# 6. TRAINING LOOP WITH CHECKPOINTING
# ============================================================
from torch.utils.data import DataLoader
from torch.optim import Adam
from tqdm.auto import tqdm

# Hyperparameters
EPOCHS = 30          # adjust as needed
BATCH_SIZE = 8       # T4 16GB handles 8-16
LR = 1e-3
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Data loaders
train_loader = DataLoader(DBSequenceDataset(split="train"), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(DBSequenceDataset(split="val"), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

optim = Adam(model.parameters(), lr=LR)

best_val_loss = float('inf')
history = {"train": [], "val": []}

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss = 0.0
    for X, y_dbz, y_lightning in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]"):
        X, y_dbz, y_lightning = X.to(device), y_dbz.to(device), y_lightning.to(device)
        optim.zero_grad()
        pred_dbz, pred_lightning = model(X)
        loss, parts = nowcast_loss(
            pred_dbz, pred_lightning, y_dbz, y_lightning,
            reflect_mean=REFLECT_MEAN, reflect_std=REFLECT_STD,
        )
        loss.backward()
        optim.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # --- Validate ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X, y_dbz, y_lightning in val_loader:
            X, y_dbz, y_lightning = X.to(device), y_dbz.to(device), y_lightning.to(device)
            pred_dbz, pred_lightning = model(X)
            loss, _ = nowcast_loss(
                pred_dbz, pred_lightning, y_dbz, y_lightning,
                reflect_mean=REFLECT_MEAN, reflect_std=REFLECT_STD,
            )
            val_loss += loss.item()
    val_loss /= len(val_loader)

    history["train"].append(train_loss)
    history["val"].append(val_loss)

    # --- Checkpoint ---
    ckpt = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optim_state_dict": optim.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "reflect_mean": REFLECT_MEAN,
        "reflect_std": REFLECT_STD,
    }
    torch.save(ckpt, CHECKPOINT_DIR / f"vajra_epoch_{epoch:03d}.pt")

    # Save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(ckpt, CHECKPOINT_DIR / "vajra_best.pt")
        best_mark = " BEST"
    else:
        best_mark = ""

    print(f"Epoch {epoch:3d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}{best_mark}")

# Save final
torch.save(model.state_dict(), "vajra_nowcast.pt")
print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
print(f"Checkpoints in: {CHECKPOINT_DIR.resolve()}")

In [ ]:
# ============================================================
# 7. PLOT TRAINING CURVES
# ============================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(history["train"], label="Train")
plt.plot(history["val"], label="Val")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE + 2xBCE)")
plt.title("VAJRA Training Curves")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# 8. EVALUATION ON TEST SPLIT
# ============================================================
import numpy as np
from evaluate import scores_by_lead_time

# Load best model
ckpt = torch.load(CHECKPOINT_DIR / "vajra_best.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Run inference on test set
test_loader = DataLoader(DBSequenceDataset(split="test"), batch_size=4, shuffle=False)

all_pred_dbz = []
all_true_dbz = []
all_pred_lightning = []
all_true_lightning = []

with torch.no_grad():
    for X, y_dbz, y_lightning in tqdm(test_loader, desc="Test inference"):
        X = X.to(device)
        pred_dbz, pred_lightning = model(X)
        all_pred_dbz.append(pred_dbz.cpu().numpy())
        all_true_dbz.append(y_dbz.numpy())
        all_pred_lightning.append(pred_lightning.cpu().numpy())
        all_true_lightning.append(y_lightning.numpy())

pred_dbz = np.concatenate(all_pred_dbz, axis=0)      # (N, 18, 128, 128)
true_dbz = np.concatenate(all_true_dbz, axis=0)
pred_lightning = np.concatenate(all_pred_lightning, axis=0)
true_lightning = np.concatenate(all_true_lightning, axis=0)

print(f"Test samples: {pred_dbz.shape[0]}")

# Compute metrics per lead time (threshold: 35 dBZ for reflectivity, 0.5 for lightning)
print("\n=== REFLECTIVITY (thresh=35 dBZ) ===")
ref_scores = scores_by_lead_time(pred_dbz, true_dbz, threshold=35.0)
for s in ref_scores:
    print(f"  T+{s['lead_time_min']:3d} min  CSI={s['CSI']:.3f}  POD={s['POD']:.3f}  FAR={s['FAR']:.3f}  HSS={s['HSS']:.3f}")

print("\n=== LIGHTNING (thresh=0.5 prob) ===")
ltg_scores = scores_by_lead_time(pred_lightning, true_lightning, threshold=0.5)
for s in ltg_scores:
    print(f"  T+{s['lead_time_min']:3d} min  CSI={s['CSI']:.3f}  POD={s['POD']:.3f}  FAR={s['FAR']:.3f}  HSS={s['HSS']:.3f}")

# Save metrics for presentation
import json
with open("test_metrics.json", "w") as f:
    json.dump({"reflectivity": ref_scores, "lightning": ltg_scores}, f, indent=2)
print("\nSaved test_metrics.json")

In [ ]:
# ============================================================
# 9. PLOT SKILL DECAY (KEY PRESENTATION CHART)
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
metrics = ["CSI", "POD", "FAR", "HSS"]
titles = ["Critical Success Index", "Probability of Detection", "False Alarm Ratio", "Heidke Skill Score"]

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx // 2, idx % 2]

    # Reflectivity
    lead = [s["lead_time_min"] for s in ref_scores]
    vals = [s[metric] for s in ref_scores]
    ax.plot(lead, vals, 'o-', label="Reflectivity (>=35 dBZ)", color="tab:blue")

    # Lightning
    vals_ltg = [s[metric] for s in ltg_scores]
    ax.plot(lead, vals_ltg, 's--', label="Lightning (p>=0.5)", color="tab:orange")

    ax.set_xlabel("Lead Time (min)")
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)

fig.suptitle("VAJRA Nowcast Skill Decay - TEST SPLIT (SYNTHETIC DATA)", fontsize=12, y=1.02)
fig.tight_layout()
plt.savefig("skill_decay.png", dpi=150, bbox_inches='tight')
plt.show()

print("Saved skill_decay.png - use this in your presentation")

In [ ]:
# ============================================================
# 10. QUALITATIVE VISUALIZATION
# ============================================================
import matplotlib.pyplot as plt
from normalization import load_normalization_stats

CHANNEL_MEAN, CHANNEL_STD = load_normalization_stats()

# Get one test sequence
test_ds = DBSequenceDataset(split="test")
X, y_dbz, y_lightning = test_ds[0]  # (6,5,128,128), (18,128,128), (18,128,128)

# Model prediction
model.eval()
with torch.no_grad():
    pred_dbz, pred_lightning = model(X.unsqueeze(0).to(device))
pred_dbz = pred_dbz[0].cpu().numpy()      # (18, 128, 128)
pred_lightning = pred_lightning[0].cpu().numpy()

# Denormalize input radar
radar_in = X[:, 0].numpy() * CHANNEL_STD[0] + CHANNEL_MEAN[0]  # (6, 128, 128)

# Plot: 6 input + 18 forecast = 24 frames
total = 24
cols = 6
rows = 4
fig, axes = plt.subplots(rows, cols, figsize=(14, 9))
vmin, vmax = -30, 75

for idx in range(total):
    r, c = divmod(idx, cols)
    ax = axes[r, c]
    ax.set_xticks([]); ax.set_yticks([])

    if idx < 6:
        frame = radar_in[idx]
        title = f"Input t-{(6-idx)*10}m"
        color = "white"
    else:
        step = idx - 6
        frame = pred_dbz[step]
        title = f"Forecast t+{(step+1)*10}m"
        color = "yellow"

    im = ax.imshow(frame, cmap="turbo", vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=9, color=color)

# Hide extra subplots
for idx in range(total, rows*cols):
    axes.flat[idx].axis('off')

fig.suptitle("Radar Reflectivity (dBZ) - History to Forecast", fontsize=13)
fig.tight_layout()
plt.savefig("sample_forecast.png", dpi=150, facecolor='black')
plt.show()

print("Saved sample_forecast.png")

In [ ]:
# ============================================================
# 11. DOWNLOAD ARTIFACTS
# ============================================================
# Run this cell to download all important files

from google.colab import files

artifacts = [
    "vajra_nowcast.pt",
    "vajra_best.pt",
    "test_metrics.json",
    "training_curves.png",
    "skill_decay.png",
    "sample_forecast.png",
]

# Also download all epoch checkpoints
for ckpt in sorted(CHECKPOINT_DIR.glob("vajra_epoch_*.pt")):
    artifacts.append(str(ckpt))

for f in artifacts:
    if Path(f).exists():
        print(f"Downloading: {f}")
        files.download(f)
    else:
        print(f"Missing: {f}")

print("\nAll artifacts downloaded. Save them locally!")

---

## QUICK REFERENCE

| Task | Command |
|------|---------|
| Resume training | Load vajra_best.pt - continue loop |
| Evaluate different threshold | Change threshold in scores_by_lead_time |
| Plot more samples | Change test_ds[0] index in Cell 10 |
| Use for demo | Load vajra_nowcast.pt in inference script |

## IMPORTANT NOTES

1. Results are on SYNTHETIC DATA - label all charts "Synthetic Data Results" in presentation
2. Save checkpoints often - Colab disconnects without warning
3. Free T4 - 30-45 min for 30 epochs; reduce EPOCHS if time is short
4. Real data integration - swap DBSequenceDataset for real archive when available

## PRESENTATION CHECKLIST

- Skill decay curves (Cell 9) - skill_decay.png
- Sample forecast sequence (Cell 10) - sample_forecast.png
- Training curves (Cell 7) - training_curves.png
- Test metrics table (Cell 8 output) - copy to slides
- Explicit "Synthetic vs Real" slide
- Architecture diagram (ConvLSTM encoder-decoder, dual heads)

---